# modelos avançados (execução real) — modelos avançados no Kaggle

## Antes de rodar

1. **Suba `src/churn_telecom/` como um Kaggle Dataset** chamado
   `churn-telecom-src` (ou ajuste `SRC_DIR` abaixo) — é o mesmo pacote já
   testado e usado nas etapas anteriores, sem nenhuma mudança.
2. **Suba `data/telco_customer_churn.csv`** como outro Dataset chamado
   `churn-telecom-data` (ou ajuste `DATA_DIR` abaixo).
3. **Ative GPU** no notebook (Settings → Accelerator → GPU).
4. Rode as células na ordem.


In [ ]:
# Ajuste estes caminhos conforme o nome real dos datasets que voce subiu.
import sys
from pathlib import Path

ROOT = Path("/kaggle/working")
DATA_DIR = Path("/kaggle/input/churn-telecom-data")
SRC_DIR = Path("/kaggle/input/churn-telecom-src")
sys.path.insert(0, str(SRC_DIR))

(ROOT / "results").mkdir(exist_ok=True)
(ROOT / "reports" / "tables").mkdir(parents=True, exist_ok=True)
print("paths ok")


## 1. Instalação dos pacotes

O Kaggle já vem com PyTorch e a maioria das libs de ML — só falta instalar
os 5 modelos avançados em si.

In [ ]:
!pip install -q tabpfn pykan tabkan
!pip install -q "autogluon.tabular[mitra]"


## 2. Preparação dos dados (idêntica às etapas anteriores — mesmo pacote)

In [ ]:
import torch
import joblib
import numpy as np
import pandas as pd

from churn_telecom.data import load_raw, split_three_stage, assert_sem_vazamento
from churn_telecom.features import Preprocessor
from churn_telecom.metrics import full_report
from churn_telecom.checkpointing import save_run, load_all_metadata

SEED = 0
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

df = load_raw(DATA_DIR)
split = split_three_stage(df, seed=SEED)
assert_sem_vazamento(split)
df_train, df_val, df_test = df.iloc[split.train_idx], df.iloc[split.val_idx], df.iloc[split.test_idx]

prep = Preprocessor().fit(df_train)
X_train, cols = prep.transform(df_train)
X_val, _ = prep.transform(df_val)
X_test, _ = prep.transform(df_test)
y_train = prep.transform_target(df_train)
y_val = prep.transform_target(df_val)
y_test = prep.transform_target(df_test)

print(f"X_train: {X_train.shape} | X_val: {X_val.shape} | X_test: {X_test.shape}")


## 3. STab (implementação própria — ver `src/churn_telecom/models/stab.py`)

Não existe um pacote público legítimo chamado "STab" (colisão de nome no
PyPI — ver smoke test da modelos avançados). Esta é uma implementação própria e
enxuta de Transformer para dados tabulares, cobrindo o espaço de
hiperparâmetros pedido no enunciado.

In [ ]:
from churn_telecom.models.stab import STabHyperparams, treinar_stab, prever_stab

hp_stab = STabHyperparams(dim=32, depth=2, heads=4, attn_dropout=0.1, ff_dropout=0.1,
                           lr=1e-3, weight_decay=1e-5, batch_size=64, max_epochs=200, patience=10, seed=SEED)
modelo_stab, historico_stab = treinar_stab(X_train, y_train, X_val, y_val, hp_stab, device=device, verbose=True)
p_test_stab = prever_stab(modelo_stab, X_test, device=device)
metricas_stab = full_report(y_test, p_test_stab)
print(metricas_stab)

def salvar_stab(save_dir):
    torch.save(modelo_stab.state_dict(), save_dir / "model.pt")

save_run(ROOT / "results", "stab_churn", framework="PyTorch (STab — implementacao propria)",
         hyperparameters=hp_stab.__dict__, metrics=metricas_stab,
         architecture={"dim": hp_stab.dim, "depth": hp_stab.depth, "heads": hp_stab.heads},
         history=historico_stab, notes="modelos avançados — STab (implementacao propria).", save_model_fn=salvar_stab)


## 4. TabPFN v2

In [ ]:
from churn_telecom.models.tabpfn import TabPFNHyperparams, treinar_tabpfn, prever_tabpfn

hp_tabpfn = TabPFNHyperparams(device=device, n_estimators=8, seed=SEED)
modelo_tabpfn = treinar_tabpfn(X_train, y_train, hp_tabpfn)
p_test_tabpfn = prever_tabpfn(modelo_tabpfn, X_test)
metricas_tabpfn = full_report(y_test, p_test_tabpfn)
print(metricas_tabpfn)

def salvar_tabpfn(save_dir):
    joblib.dump(modelo_tabpfn, save_dir / "model.joblib")

save_run(ROOT / "results", "tabpfn_churn", framework="TabPFN v2", hyperparameters=hp_tabpfn.__dict__,
         metrics=metricas_tabpfn, notes="modelos avançados — TabPFN v2.", save_model_fn=salvar_tabpfn)


## 5. KAN

In [ ]:
from churn_telecom.models.kan import KANHyperparams, treinar_kan, prever_kan

hp_kan = KANHyperparams(hidden_width=10, grid=5, k=3, lr=1e-3, steps=50, seed=SEED)
modelo_kan, historico_kan = treinar_kan(X_train, y_train, X_val, y_val, hp_kan, device=device)
p_test_kan = prever_kan(modelo_kan, X_test, device=device)
metricas_kan = full_report(y_test, p_test_kan)
print(metricas_kan)

def salvar_kan(save_dir):
    torch.save(modelo_kan.state_dict(), save_dir / "model.pt")

save_run(ROOT / "results", "kan_churn", framework="pykan (KAN)", hyperparameters=hp_kan.__dict__,
         metrics=metricas_kan, architecture={"hidden_width": hp_kan.hidden_width, "grid": hp_kan.grid, "k": hp_kan.k},
         notes="modelos avançados — KAN.", save_model_fn=salvar_kan)


## 6. TabKAN

**Atenção:** a API exata do pacote `tabkan` não pôde ser inspecionada no
ambiente local (não instala lá). Se `TabKANClassifier` não existir ou
tiver assinatura diferente, ajuste `src/churn_telecom/models/tabkan.py`
conforme a documentação real do pacote instalado aqui.

In [ ]:
from churn_telecom.models.tabkan import TabKANHyperparams, treinar_tabkan, prever_tabkan

hp_tabkan = TabKANHyperparams(hidden_width=10, grid=5, k=3, lr=1e-3, epochs=100, seed=SEED)
modelo_tabkan = treinar_tabkan(X_train, y_train, hp_tabkan)
p_test_tabkan = prever_tabkan(modelo_tabkan, X_test)
metricas_tabkan = full_report(y_test, p_test_tabkan)
print(metricas_tabkan)

def salvar_tabkan(save_dir):
    joblib.dump(modelo_tabkan, save_dir / "model.joblib")

save_run(ROOT / "results", "tabkan_churn", framework="tabkan", hyperparameters=hp_tabkan.__dict__,
         metrics=metricas_tabkan, notes="modelos avançados — TabKAN.", save_model_fn=salvar_tabkan)


## 7. Mitra (via AutoGluon)

In [ ]:
from churn_telecom.models.mitra import MitraHyperparams, treinar_mitra, prever_mitra

hp_mitra = MitraHyperparams(time_limit_segundos=600, seed=SEED)
preditor_mitra = treinar_mitra(X_train, y_train, cols, hp_mitra)
p_test_mitra = prever_mitra(preditor_mitra, X_test, cols)
metricas_mitra = full_report(y_test, p_test_mitra)
print(metricas_mitra)

def salvar_mitra(save_dir):
    preditor_mitra.save(path=str(save_dir / "autogluon_predictor"))

save_run(ROOT / "results", "mitra_churn", framework="AutoGluon (extra [mitra])",
         hyperparameters=hp_mitra.__dict__, metrics=metricas_mitra, notes="modelos avançados — Mitra.",
         save_model_fn=salvar_mitra)


## 8. Comparação final — todos os modelos do projeto

Esta célula também funciona se algum dos modelos acima falhar (está isolado
célula a célula) — basta ter rodado os que funcionaram.

In [ ]:
resumo = load_all_metadata(ROOT / "results")
cols_mostrar = ["model_id", "metrics.ks", "metrics.auroc", "metrics.recall", "metrics.precision"]
cols_existentes = [c for c in cols_mostrar if c in resumo.columns]
tabela = resumo[cols_existentes].sort_values("metrics.ks", ascending=False).reset_index(drop=True)
tabela


## 9. Baixando os resultados de volta

Depois de rodar, baixe a pasta `/kaggle/working/results/` (Kaggle permite
baixar o diretório de output do notebook) e cole em `results/` no projeto
local, junto com `reports/tables/modelos_avancados_resumo_kaggle.csv` — o notebook de
consolidação final só lê `results/`, não retreina nada.